In [52]:
# import libraries

import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.agents import create_agent

In [9]:
load_dotenv()

True

In [10]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

In [11]:
# Loading data
DATA_FILE_PATH = os.path.join("data","hr_policy.txt")
print(DATA_FILE_PATH)

data\hr_policy.txt


In [12]:
# Data Injestion

loader = TextLoader(DATA_FILE_PATH,encoding="utf-8")

documents = loader.load()

print("Data Loaded")
print("="*40)
print(documents)

Data Loaded
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

### Langchain documents

Langchain processes everything in the form of documents.

DOCUMENTS:

PAGE CONTENT -- Actual Data

METADATA -- Extra Information About The Data



In [13]:
len(documents)

1

In [15]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [16]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [17]:
print(f'Total number of characters in document: {len(documents[0].page_content)}')

Total number of characters in document: 2598


In [23]:
# Splitting Data

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)

print(chunks[0:2])

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.')]


In [ ]:
# Now each splitted chunk is document and each document will have page content and metadata.
len(chunks)

9

In [27]:
print(chunks[0])
print("="*40)
print(chunks[1])
print("="*40)
print(chunks[2])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}
page_content='1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.' metadata={'source': 'data\\hr_policy.txt'}
page_content='2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.' metadata={'source': 'data\\hr_policy.txt'}


In [31]:
# Embed data

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print(f'Embeddings model is ready and the name is {embeddings_model.model_name}')

Embeddings model is ready and the name is jina-embeddings-v2-base-en


In [35]:
# Store data in vector database

vector_store = FAISS.from_documents(chunks,embeddings_model)

print("Chunks are stored",vector_store.index.ntotal)

Chunks are stored 9


In [36]:
test_query = "How many sick leaves an employee gets?"

# Similarity Search

top_matches = vector_store.similarity_search(test_query,k=2)

print(f"Query: {test_query}\n")

Query: How many sick leaves an employee gets?



In [39]:
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [49]:
# LLM

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature = 0.7
)

llm.model_name

'openai/gpt-oss-120b'

In [50]:
test_response = llm.invoke("Is learning Gen AI hard? Answer in one line")
test_response.content

'It can be challenging at first, but with focused resources and hands‑on practice, most people pick it up fairly quickly.'

In [51]:
test_response = llm.invoke("Is learning RAG hard?Answer in one line")
test_response.content

'Learning RAG can be challenging at first, but with clear resources and hands‑on practice it becomes manageable.'

### AI Agent

3 --

LLM - BRAIN
TOOL - Super Power
Memory - No memory currently


In [59]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question:str) ->str:
    """ 
        Search the HR policy document for information about leave,work from home,probation,notice period,reimbursement,
        code of conduct,holidays,or exit process.
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [60]:
hr_assistant = create_agent(
    model = llm,
    tools = [search_hr_policy],
    system_prompt="""
               You are a friendly HR assistant.
               Always use the search_hr_policy tool to look up facts before answering.
               If the answer is not in the search results,say you don't know instead of guessing.
    """
)

In [61]:
def ask_hr_assistant(question:str)->str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:",question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages":[{"role":"user","content":question}]})
    answer = response["message"][-1].content

    print("ANSWER:",answer)
    print("=" * 60)
    print()

    return answer

In [62]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"tell me about leave policies how to apply for leave"
            }
        ]
    }
)

In [64]:
response["messages"][-1].content

'**Leave Policies (as per the company HR policy)**  \n\n| Type of leave | Entitlement | Key rules |\n|---------------|------------|-----------|\n| **Annual (paid) leave** | 20\u202fdays per calendar year | • Requests must be submitted through the HR portal **at least\u202f5\u202fworking days** before the intended start date.<br>• Up to **5\u202funused days** can be carried forward to the next year; any surplus is forfeited. |\n| **Sick leave** | 10\u202fpaid days per year | • Separate from annual leave.<br>• A medical certificate is required **if the sick leave exceeds 2 consecutive days**. |\n| **Public holidays** | 12\u202fdays per year (as per the official holiday calendar) | • If you work on a public holiday you are eligible for compensatory leave. |\n\n**How to apply for leave**\n\n1. **Log into the HR portal** – Use your employee credentials to access the “Leave Management” section.  \n2. **Select the leave type** – Choose *Annual Leave* or *Sick Leave* (or any other applicable c

In [65]:
print(response["messages"][-1].content)

**Leave Policies (as per the company HR policy)**  

| Type of leave | Entitlement | Key rules |
|---------------|------------|-----------|
| **Annual (paid) leave** | 20 days per calendar year | • Requests must be submitted through the HR portal **at least 5 working days** before the intended start date.<br>• Up to **5 unused days** can be carried forward to the next year; any surplus is forfeited. |
| **Sick leave** | 10 paid days per year | • Separate from annual leave.<br>• A medical certificate is required **if the sick leave exceeds 2 consecutive days**. |
| **Public holidays** | 12 days per year (as per the official holiday calendar) | • If you work on a public holiday you are eligible for compensatory leave. |

**How to apply for leave**

1. **Log into the HR portal** – Use your employee credentials to access the “Leave Management” section.  
2. **Select the leave type** – Choose *Annual Leave* or *Sick Leave* (or any other applicable category).  
3. **Enter dates** – Specify t

In [66]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"Which organization do you work for?"
            }
        ]
    }
)

print(response["messages"][-1].content)

I’m part of the HR team at **Acme Corp**.
